# HW1: Multimodal Data Preprocessing — EPIC-KITCHENS Action Recognition

**Course:** Modelling: Multimodal AI  
**Task:** Egocentric kitchen action recognition from video (predict verb + noun action labels)  
**Dataset:** EPIC-KITCHENS-100  
**Modalities:** RGB frames, audio (spectrograms/MFCCs), optical flow, text narrations


---
## Part 0: Setup & Installation


In [ ]:
# Install required packages
!pip install -q opencv-python-headless librosa soundfile scikit-learn matplotlib seaborn pandas numpy

# For audio extraction from video
!apt-get -qq install ffmpeg


In [ ]:
import cv2
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import subprocess
import warnings
from pathlib import Path
from sklearn.manifold import TSNE
from collections import Counter

warnings.filterwarnings('ignore')

# Create directory structure
BASE_DIR = "epic_kitchens_hw1"
os.makedirs(f"{BASE_DIR}/videos", exist_ok=True)
os.makedirs(f"{BASE_DIR}/frames", exist_ok=True)
os.makedirs(f"{BASE_DIR}/audio", exist_ok=True)
os.makedirs(f"{BASE_DIR}/optical_flow", exist_ok=True)
os.makedirs(f"{BASE_DIR}/annotations", exist_ok=True)

print("Setup complete!")


---
## Part 1: Data Download

We download annotation files from the EPIC-KITCHENS-100 GitHub repo and a small
subset of videos from participant P01. The full dataset is ~740GB; we work with
just 2–3 videos to keep this Colab-friendly.


In [ ]:
# Download annotation CSVs from EPIC-KITCHENS-100 annotations repo
import urllib.request

annotation_urls = {
    "EPIC_100_train.csv": "https://raw.githubusercontent.com/epic-kitchens/epic-kitchens-100-annotations/master/EPIC_100_train.csv",
    "EPIC_100_validation.csv": "https://raw.githubusercontent.com/epic-kitchens/epic-kitchens-100-annotations/master/EPIC_100_validation.csv",
}

for fname, url in annotation_urls.items():
    out_path = f"{BASE_DIR}/annotations/{fname}"
    if not os.path.exists(out_path):
        print(f"Downloading {fname}...")
        urllib.request.urlretrieve(url, out_path)
        print(f"  -> Saved to {out_path}")
    else:
        print(f"{fname} already exists, skipping.")

# Load and inspect annotations
train_df = pd.read_csv(f"{BASE_DIR}/annotations/EPIC_100_train.csv")
print(f"\nTraining annotations shape: {train_df.shape}")
print(f"Columns: {list(train_df.columns)}")
train_df.head()


In [ ]:
# Filter annotations for participant P01 (the subset we'll download videos for)
p01_df = train_df[train_df["participant_id"] == "P01"].copy()
print(f"P01 annotations: {p01_df.shape[0]} action segments")
print(f"P01 videos: {p01_df['video_id'].unique()}")
print(f"\nSample annotations:")
print(p01_df[["narration", "verb", "noun", "verb_class", "noun_class", "start_frame", "stop_frame"]].head(15))


In [ ]:
# Download 2 videos from P01 using the official download scripts
# NOTE: The data.bris server can be slow. If downloads fail, you can also:
# 1. Download manually from https://data.bris.ac.uk/data/dataset/2g1n6qdydwa9u22shpxqzp0t8m
# 2. Use Academic Torrents
# For this notebook, we use the official downloader.

!git clone --quiet https://github.com/epic-kitchens/epic-kitchens-download-scripts.git 2>/dev/null || true

# Download just 2 specific videos from P01
# These are shorter videos that are manageable for Colab
!cd epic-kitchens-download-scripts && python epic_downloader.py \
    --videos \
    --specific-videos P01_101,P01_102 \
    --output-path ../{BASE_DIR}/

# Check what was downloaded
video_dir = f"{BASE_DIR}/P01/videos"
if os.path.exists(video_dir):
    videos = os.listdir(video_dir)
    print(f"Downloaded videos: {videos}")
    for v in videos:
        size_mb = os.path.getsize(f"{video_dir}/{v}") / (1024*1024)
        print(f"  {v}: {size_mb:.1f} MB")
else:
    print("Videos not yet downloaded. See manual download instructions above.")
    # Fallback: create the directory so downstream code can check
    os.makedirs(video_dir, exist_ok=True)


In [ ]:
# FALLBACK: If the official downloader is slow or fails, you can download
# directly via wget. Uncomment and run the lines below:

# !wget -q -P {BASE_DIR}/P01/videos/ \
#     "https://data.bris.ac.uk/datasets/2g1n6qdydwa9u22shpxqzp0t8m/P01/videos/P01_101.MP4"
# !wget -q -P {BASE_DIR}/P01/videos/ \
#     "https://data.bris.ac.uk/datasets/2g1n6qdydwa9u22shpxqzp0t8m/P01/videos/P01_102.MP4"

# List available videos
video_dir = f"{BASE_DIR}/P01/videos"
available_videos = sorted([f for f in os.listdir(video_dir) if f.endswith(".MP4")]) if os.path.exists(video_dir) else []
print(f"Available videos: {available_videos}")


---
## Part 2: Modality Extraction (20 pts)

We extract four modalities from the EPIC-KITCHENS data:

1. **RGB Frames** — visual scene information (spatial "where")
2. **Audio** — spectrograms and MFCCs capturing interaction sounds (temporal "when" + contact info)
3. **Optical Flow** — motion patterns between consecutive frames (dynamic "how")
4. **Text Narrations** — natural language descriptions of actions (semantic "what")

Each modality provides *unique* information that the others cannot fully capture.


### 2.1 RGB Frame Extraction

Extract frames from video at a fixed sampling rate. EPIC-KITCHENS videos are
recorded at ~60fps; we subsample to reduce storage while retaining visual diversity.


In [ ]:
def extract_frames(video_path, output_dir, sample_rate=10, max_frames=500):
    """
    Extract RGB frames from a video file at a given sampling rate.

    Args:
        video_path (str): Path to the video file.
        output_dir (str): Directory to save extracted frames.
        sample_rate (int): Extract every Nth frame (e.g., 10 = every 10th frame).
        max_frames (int): Maximum number of frames to extract.

    Returns:
        list: Paths to extracted frame files.
    """
    os.makedirs(output_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open video {video_path}")
        return []

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Video: {os.path.basename(video_path)}")
    print(f"  FPS: {fps:.1f}, Total frames: {total_frames}, Duration: {total_frames/fps:.1f}s")
    print(f"  Sampling every {sample_rate}th frame -> ~{total_frames // sample_rate} frames")

    extracted_paths = []
    frame_idx = 0
    saved_count = 0

    while True:
        ret, frame = cap.read()
        if not ret or saved_count >= max_frames:
            break

        if frame_idx % sample_rate == 0:
            # Resize to a manageable resolution (256x256 for feature extraction)
            frame_resized = cv2.resize(frame, (256, 256))
            frame_filename = os.path.join(output_dir, f"frame_{frame_idx:06d}.jpg")
            cv2.imwrite(frame_filename, frame_resized)
            extracted_paths.append(frame_filename)
            saved_count += 1

        frame_idx += 1

    cap.release()
    print(f"  Extracted {saved_count} frames to {output_dir}")
    return extracted_paths


# Extract frames from each available video
all_frame_paths = {}
for video_file in available_videos:
    video_id = video_file.replace(".MP4", "")
    video_path = f"{BASE_DIR}/P01/videos/{video_file}"
    out_dir = f"{BASE_DIR}/frames/{video_id}"

    paths = extract_frames(video_path, out_dir, sample_rate=15, max_frames=300)
    all_frame_paths[video_id] = paths
    print()

# Show a sample grid of extracted frames
if all_frame_paths:
    first_vid = list(all_frame_paths.keys())[0]
    sample_paths = all_frame_paths[first_vid][:12]

    fig, axes = plt.subplots(2, 6, figsize=(18, 6))
    fig.suptitle(f"Sample RGB Frames from {first_vid}", fontsize=14)
    for ax, fpath in zip(axes.flat, sample_paths):
        img = cv2.imread(fpath)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img_rgb)
        ax.set_title(os.path.basename(fpath).replace("frame_", "").replace(".jpg", ""), fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


### 2.2 Audio Extraction (Spectrograms & MFCCs)

We extract the audio track from each video using `ffmpeg`, then compute:
- **Mel Spectrograms**: frequency-time representations that capture the character of sounds
- **MFCCs**: compact cepstral coefficients commonly used as audio features

Audio is critical for kitchen actions — the sound of chopping, water running, or a
microwave beeping provides information not available from vision alone.


In [ ]:
def extract_audio(video_path, output_path, sr=22050):
    """
    Extract audio from video file using ffmpeg.

    Args:
        video_path (str): Path to input video.
        output_path (str): Path to save .wav file.
        sr (int): Target sample rate.

    Returns:
        tuple: (audio_array, sample_rate) or (None, None) on failure.
    """
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # Use ffmpeg to extract audio as WAV
    cmd = [
        "ffmpeg", "-i", video_path,
        "-vn",                    # no video
        "-acodec", "pcm_s16le",   # PCM 16-bit
        "-ar", str(sr),           # sample rate
        "-ac", "1",               # mono
        "-y",                     # overwrite
        output_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(f"  Warning: ffmpeg failed for {video_path}")
        print(f"  stderr: {result.stderr[:200]}")
        return None, None

    # Load with librosa
    y, sr_out = librosa.load(output_path, sr=sr)
    print(f"  Audio: {len(y)/sr_out:.1f}s, sample rate={sr_out}")
    return y, sr_out


def compute_audio_features(y, sr, n_mfcc=13, hop_length=512):
    """
    Compute mel spectrogram and MFCCs from audio.

    Args:
        y (np.array): Audio time series.
        sr (int): Sample rate.
        n_mfcc (int): Number of MFCC coefficients.
        hop_length (int): Hop length for STFT.

    Returns:
        dict with 'mel_spectrogram', 'mfcc', 'mfcc_delta', 'mfcc_delta2'
    """
    # Mel spectrogram
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, hop_length=hop_length, n_mels=128)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

    # MFCCs and their deltas
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, hop_length=hop_length)
    mfcc_delta = librosa.feature.delta(mfcc)
    mfcc_delta2 = librosa.feature.delta(mfcc, order=2)

    return {
        "mel_spectrogram": mel_spec_db,
        "mfcc": mfcc,
        "mfcc_delta": mfcc_delta,
        "mfcc_delta2": mfcc_delta2,
    }


# Extract audio and compute features for each video
audio_features = {}
for video_file in available_videos:
    video_id = video_file.replace(".MP4", "")
    video_path = f"{BASE_DIR}/P01/videos/{video_file}"
    audio_path = f"{BASE_DIR}/audio/{video_id}.wav"

    print(f"Processing audio for {video_id}...")
    y, sr = extract_audio(video_path, audio_path)

    if y is not None:
        features = compute_audio_features(y, sr)
        audio_features[video_id] = {"audio": y, "sr": sr, **features}
        print(f"  Mel spectrogram shape: {features['mel_spectrogram'].shape}")
        print(f"  MFCC shape: {features['mfcc'].shape}")
    print()

# Visualize audio features for the first video
if audio_features:
    vid_id = list(audio_features.keys())[0]
    feat = audio_features[vid_id]

    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    fig.suptitle(f"Audio Features — {vid_id}", fontsize=14)

    # Mel spectrogram
    librosa.display.specshow(feat["mel_spectrogram"], sr=feat["sr"],
                             hop_length=512, x_axis="time", y_axis="mel", ax=axes[0])
    axes[0].set_title("Mel Spectrogram (dB)")
    axes[0].set_xlabel("")
    plt.colorbar(axes[0].collections[0], ax=axes[0], format="%+2.0f dB")

    # MFCCs
    librosa.display.specshow(feat["mfcc"], sr=feat["sr"],
                             hop_length=512, x_axis="time", ax=axes[1])
    axes[1].set_title(f"MFCCs ({feat['mfcc'].shape[0]} coefficients)")
    plt.colorbar(axes[1].collections[0], ax=axes[1])

    plt.tight_layout()
    plt.show()


### 2.3 Optical Flow Extraction

Optical flow captures motion between consecutive frames. We use the Farneback
dense optical flow algorithm from OpenCV. This modality is unique because it
encodes *dynamics* — how things are moving — which static RGB frames don't capture.


In [ ]:
def compute_optical_flow(frame_paths, output_dir, max_pairs=100):
    """
    Compute dense optical flow between consecutive frames using Farneback method.

    Args:
        frame_paths (list): Sorted list of frame image paths.
        output_dir (str): Directory to save flow visualizations.
        max_pairs (int): Maximum number of frame pairs to process.

    Returns:
        list of np.array: Flow magnitude vectors for each pair.
    """
    os.makedirs(output_dir, exist_ok=True)

    flow_magnitudes = []
    pairs_to_process = min(len(frame_paths) - 1, max_pairs)

    for i in range(pairs_to_process):
        img1 = cv2.imread(frame_paths[i])
        img2 = cv2.imread(frame_paths[i + 1])

        gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
        gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

        # Compute dense optical flow
        flow = cv2.calcOpticalFlowFarneback(
            gray1, gray2, None,
            pyr_scale=0.5, levels=3, winsize=15,
            iterations=3, poly_n=5, poly_sigma=1.2, flags=0
        )

        # Compute magnitude and angle
        mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
        flow_magnitudes.append(mag.mean())

        # Save visualization using HSV encoding (every 10th pair)
        if i % 10 == 0:
            hsv = np.zeros_like(img1)
            hsv[..., 0] = ang * 180 / np.pi / 2    # Hue = direction
            hsv[..., 1] = 255                        # Full saturation
            hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)  # Value = magnitude
            flow_rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
            cv2.imwrite(f"{output_dir}/flow_{i:05d}.jpg", flow_rgb)

    print(f"  Computed flow for {pairs_to_process} frame pairs")
    return flow_magnitudes


# Compute optical flow for each video
flow_data = {}
for video_id, frame_paths in all_frame_paths.items():
    if len(frame_paths) < 2:
        continue
    print(f"Computing optical flow for {video_id}...")
    out_dir = f"{BASE_DIR}/optical_flow/{video_id}"
    magnitudes = compute_optical_flow(frame_paths, out_dir, max_pairs=200)
    flow_data[video_id] = magnitudes
    print(f"  Mean flow magnitude: {np.mean(magnitudes):.3f}")
    print()

# Visualize optical flow samples
if flow_data:
    vid_id = list(flow_data.keys())[0]
    flow_dir = f"{BASE_DIR}/optical_flow/{vid_id}"
    flow_imgs = sorted([f for f in os.listdir(flow_dir) if f.endswith(".jpg")])[:6]

    if flow_imgs:
        fig, axes = plt.subplots(1, len(flow_imgs), figsize=(18, 3))
        fig.suptitle(f"Optical Flow Visualizations — {vid_id}", fontsize=14)
        for ax, fname in zip(axes.flat if len(flow_imgs) > 1 else [axes], flow_imgs):
            img = cv2.imread(f"{flow_dir}/{fname}")
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img_rgb)
            ax.set_title(fname.replace("flow_", "").replace(".jpg", ""), fontsize=8)
            ax.axis("off")
        plt.tight_layout()
        plt.show()


### 2.4 Text Narration Processing

EPIC-KITCHENS provides natural language narrations for each action segment
(e.g., "open fridge", "take butter", "close drawer"). We extract and tokenize these
as a text modality, along with structured verb/noun class labels.


In [ ]:
def process_narrations(annotations_df, video_ids):
    """
    Extract and process text narrations for given video IDs.

    Args:
        annotations_df (pd.DataFrame): Full annotation dataframe.
        video_ids (list): List of video IDs to filter.

    Returns:
        pd.DataFrame: Filtered and enriched narrations dataframe.
    """
    # Filter to our videos
    narr_df = annotations_df[annotations_df["video_id"].isin(video_ids)].copy()

    # Create action label (verb + noun)
    narr_df["action"] = narr_df["verb"] + " " + narr_df["noun"]

    # Basic text statistics
    narr_df["narration_length"] = narr_df["narration"].str.split().str.len()
    narr_df["segment_duration_frames"] = narr_df["stop_frame"] - narr_df["start_frame"]

    return narr_df


# Process narrations for our videos
video_ids = [v.replace(".MP4", "") for v in available_videos]
narrations_df = process_narrations(train_df, video_ids)

print(f"Total narrated action segments: {len(narrations_df)}")
print(f"\nUnique verbs: {narrations_df['verb'].nunique()}")
print(f"Unique nouns: {narrations_df['noun'].nunique()}")
print(f"Unique actions: {narrations_df['action'].nunique()}")

print(f"\nTop 10 verbs:")
print(narrations_df["verb"].value_counts().head(10).to_string())

print(f"\nTop 10 nouns:")
print(narrations_df["noun"].value_counts().head(10).to_string())

print(f"\nSample narrations:")
print(narrations_df[["narration", "verb", "noun", "action", "start_frame", "stop_frame"]].head(10).to_string())


In [ ]:
# Simple tokenization of narrations (word-level)
from collections import Counter

all_narrations = narrations_df["narration"].dropna().tolist()
all_words = []
for narr in all_narrations:
    all_words.extend(narr.lower().split())

word_counts = Counter(all_words)
vocab_size = len(word_counts)
print(f"Vocabulary size: {vocab_size}")
print(f"\nTop 20 words: {word_counts.most_common(20)}")

# Create simple word-to-index mapping
word2idx = {word: idx for idx, (word, _) in enumerate(word_counts.most_common())}
word2idx["<PAD>"] = len(word2idx)
word2idx["<UNK>"] = len(word2idx)

def tokenize_narration(narration, word2idx, max_len=10):
    """Convert narration to integer token sequence."""
    tokens = narration.lower().split()
    indices = [word2idx.get(t, word2idx["<UNK>"]) for t in tokens]
    # Pad or truncate
    if len(indices) < max_len:
        indices += [word2idx["<PAD>"]] * (max_len - len(indices))
    else:
        indices = indices[:max_len]
    return indices

# Tokenize all narrations
narrations_df["token_ids"] = narrations_df["narration"].apply(
    lambda x: tokenize_narration(str(x), word2idx)
)

print(f"\nExample tokenization:")
for _, row in narrations_df.head(5).iterrows():
    print(f"  '{row['narration']}' -> {row['token_ids']}")


### 2.5 Summary: Extracted Modalities

| Modality | Source | Representation | Unique Information |
|----------|--------|---------------|-------------------|
| RGB Frames | Video frames (sampled) | 256×256 images | Scene layout, object appearance, spatial relations |
| Audio | Video audio track (ffmpeg) | Mel spectrograms, MFCCs | Interaction sounds (chopping, pouring), ambient noise |
| Optical Flow | Consecutive frame pairs | Dense flow magnitude/direction | Motion dynamics, speed of hand movements |
| Text Narrations | Annotation CSV | Token sequences | Semantic intent ("open fridge"), action label |


---
## Part 3: Visualizations (15 pts)

We visualize the dataset in three ways:
1. **Data Distribution** — class distribution of actions
2. **Sample Visualization** — display representative samples from each modality
3. **Input Distribution (t-SNE)** — embed features into 2D to see clustering structure


### 3.1 Visualizing Data Distribution

In [ ]:
def visualize_data_distribution(data, x_feature="X", y_feature="Y",
                                num_components=2, perplexity=30, num_iterations=300):
    """
    Visualize the distribution of data using t-SNE reduction and KDE plot.

    Args:
        data (np.array): 2D array of shape (n_samples, n_features).
        x_feature (str): Label for x-axis.
        y_feature (str): Label for y-axis.
        num_components (int): t-SNE output dimensions.
        perplexity (float): t-SNE perplexity parameter.
        num_iterations (int): t-SNE iterations.
    """
    if len(data) < perplexity:
        perplexity = max(2, len(data) - 1)

    tsne = TSNE(n_components=num_components, perplexity=perplexity,
                n_iter=num_iterations, random_state=42)
    tsne_data = tsne.fit_transform(data)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 2D scatter
    axes[0].scatter(tsne_data[:, 0], tsne_data[:, 1], alpha=0.5, s=10)
    axes[0].set_title("t-SNE 2D Scatter")
    axes[0].set_xlabel(x_feature)
    axes[0].set_ylabel(y_feature)

    # KDE plot for each dimension
    sns.kdeplot(tsne_data[:, 0], ax=axes[1], fill=True)
    axes[1].set_title(f"Distribution of t-SNE Dim 1")
    axes[1].set_xlabel(x_feature)

    sns.kdeplot(tsne_data[:, 1], ax=axes[2], fill=True)
    axes[2].set_title(f"Distribution of t-SNE Dim 2")
    axes[2].set_xlabel(y_feature)

    plt.tight_layout()
    plt.show()

    return tsne_data


In [ ]:
# Visualize CLASS distribution (verb and noun frequencies)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Verb distribution
verb_counts = narrations_df["verb"].value_counts().head(20)
axes[0].barh(range(len(verb_counts)), verb_counts.values)
axes[0].set_yticks(range(len(verb_counts)))
axes[0].set_yticklabels(verb_counts.index, fontsize=9)
axes[0].set_xlabel("Count")
axes[0].set_title("Top 20 Verb Classes")
axes[0].invert_yaxis()

# Noun distribution
noun_counts = narrations_df["noun"].value_counts().head(20)
axes[1].barh(range(len(noun_counts)), noun_counts.values)
axes[1].set_yticks(range(len(noun_counts)))
axes[1].set_yticklabels(noun_counts.index, fontsize=9)
axes[1].set_xlabel("Count")
axes[1].set_title("Top 20 Noun Classes")
axes[1].invert_yaxis()

plt.suptitle("Action Label Distribution (P01 Subset)", fontsize=14)
plt.tight_layout()
plt.show()

# Action duration distribution
fig, ax = plt.subplots(figsize=(10, 4))
durations = narrations_df["segment_duration_frames"].dropna()
ax.hist(durations, bins=50, edgecolor="black", alpha=0.7)
ax.set_xlabel("Segment Duration (frames)")
ax.set_ylabel("Count")
ax.set_title("Distribution of Action Segment Durations")
ax.axvline(durations.median(), color='red', linestyle='--', label=f"Median: {durations.median():.0f}")
ax.legend()
plt.tight_layout()
plt.show()


### 3.2 Visualizing Samples

In [ ]:
def visualize_samples(data, labels=None, num_samples=10, title="Random Samples"):
    """
    Visualize random samples from a feature array using t-SNE.

    Args:
        data (np.array): Feature array of shape (n_samples, n_features).
        labels (array-like): Optional labels for coloring.
        num_samples (int): Number of samples to highlight.
        title (str): Plot title.
    """
    if num_samples > len(data):
        print(f"Warning: num_samples ({num_samples}) > dataset size ({len(data)}). Using all.")
        num_samples = len(data)

    # Random subset indices
    sample_idx = np.random.choice(len(data), size=num_samples, replace=False)
    sample_data = data[sample_idx]

    if labels is not None:
        sample_labels = np.array(labels)[sample_idx]
    else:
        sample_labels = None

    perplexity = min(30, max(2, num_samples - 1))
    tsne = TSNE(n_components=2, perplexity=perplexity, n_iter=500, random_state=42)
    embedded = tsne.fit_transform(sample_data)

    plt.figure(figsize=(10, 7))
    if sample_labels is not None:
        unique_labels = list(set(sample_labels))
        colors = plt.cm.tab20(np.linspace(0, 1, len(unique_labels)))
        for i, label in enumerate(unique_labels):
            mask = sample_labels == label
            plt.scatter(embedded[mask, 0], embedded[mask, 1],
                       c=[colors[i]], label=label, s=40, alpha=0.7)
        plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=7)
    else:
        plt.scatter(embedded[:, 0], embedded[:, 1], alpha=0.6, s=40)

    plt.title(f"t-SNE of {title} (n={num_samples})")
    plt.xlabel("t-SNE Dim 1")
    plt.ylabel("t-SNE Dim 2")
    plt.tight_layout()
    plt.show()


In [ ]:
# Visualize multimodal sample grid: show aligned frame + flow + narration
if all_frame_paths and narrations_df is not None and len(narrations_df) > 0:
    first_vid = list(all_frame_paths.keys())[0]
    vid_narrations = narrations_df[narrations_df["video_id"] == first_vid].head(6)

    fig, axes = plt.subplots(2, 6, figsize=(20, 7))
    fig.suptitle(f"Aligned Multimodal Samples from {first_vid}", fontsize=14)

    for col_idx, (_, row) in enumerate(vid_narrations.iterrows()):
        if col_idx >= 6:
            break
        # Find the closest extracted frame to the start_frame
        target_frame = row["start_frame"]
        frame_paths = all_frame_paths[first_vid]
        frame_indices = [int(os.path.basename(p).replace("frame_", "").replace(".jpg", ""))
                        for p in frame_paths]
        closest_idx = min(range(len(frame_indices)),
                         key=lambda i: abs(frame_indices[i] - target_frame))

        # RGB frame
        img = cv2.imread(frame_paths[closest_idx])
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[0, col_idx].imshow(img_rgb)
        axes[0, col_idx].set_title(f"Frame {frame_indices[closest_idx]}", fontsize=8)
        axes[0, col_idx].axis("off")

        # Narration as text panel
        axes[1, col_idx].text(0.5, 0.5, f'"{row["narration"]}"\n\nverb: {row["verb"]}\nnoun: {row["noun"]}',
                              ha="center", va="center", fontsize=9,
                              transform=axes[1, col_idx].transAxes,
                              bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow"))
        axes[1, col_idx].axis("off")

    plt.tight_layout()
    plt.show()


### 3.3 Visualizing Input Distribution (t-SNE)

We extract feature vectors from each modality and use t-SNE to project them
into 2D for visualization. This helps reveal whether different action classes
form separable clusters in each modality's feature space.


In [ ]:
def extract_frame_features(frame_paths, max_frames=200):
    """
    Extract simple visual features from frames.
    Uses flattened, downsampled pixel values + color histograms.
    (In practice, you'd use a pretrained CNN like ResNet.)

    Args:
        frame_paths (list): Paths to frame images.
        max_frames (int): Maximum number of frames to process.

    Returns:
        np.array: Feature matrix of shape (n_frames, n_features).
    """
    features = []
    paths_used = frame_paths[:max_frames]

    for fpath in paths_used:
        img = cv2.imread(fpath)
        if img is None:
            continue

        # Resize to small thumbnail for feature vector
        thumb = cv2.resize(img, (32, 32))
        flat_pixels = thumb.flatten().astype(np.float32) / 255.0

        # Color histogram features (more compact and informative)
        hist_features = []
        for ch in range(3):
            hist = cv2.calcHist([img], [ch], None, [32], [0, 256])
            hist = hist.flatten() / hist.sum()  # Normalize
            hist_features.extend(hist)

        # Combine: color histogram (96-d) + downsampled pixels (3072-d) -> use just histogram
        features.append(np.array(hist_features))

    return np.array(features), paths_used


# Extract visual features
print("Extracting visual features for t-SNE...")
all_visual_features = []
all_visual_labels = []

for video_id, frame_paths in all_frame_paths.items():
    features, used_paths = extract_frame_features(frame_paths, max_frames=150)
    if len(features) == 0:
        continue

    # Assign action labels to frames based on annotation timestamps
    vid_annotations = narrations_df[narrations_df["video_id"] == video_id]

    for i, fpath in enumerate(used_paths):
        frame_num = int(os.path.basename(fpath).replace("frame_", "").replace(".jpg", ""))
        # Find which action segment this frame belongs to
        matching = vid_annotations[
            (vid_annotations["start_frame"] <= frame_num) &
            (vid_annotations["stop_frame"] >= frame_num)
        ]
        if len(matching) > 0:
            label = matching.iloc[0]["verb"]
        else:
            label = "none"

        all_visual_features.append(features[i])
        all_visual_labels.append(label)

visual_features_arr = np.array(all_visual_features)
print(f"Visual feature matrix shape: {visual_features_arr.shape}")
print(f"Label distribution: {Counter(all_visual_labels).most_common(10)}")


In [ ]:
# t-SNE visualization of VISUAL features colored by verb class
if len(visual_features_arr) > 10:
    # Filter to top verb classes for cleaner visualization
    top_verbs = [v for v, c in Counter(all_visual_labels).most_common(8) if v != "none"]
    mask = [l in top_verbs for l in all_visual_labels]
    filtered_features = visual_features_arr[mask]
    filtered_labels = [l for l, m in zip(all_visual_labels, mask) if m]

    if len(filtered_features) > 10:
        perp = min(30, len(filtered_features) - 1)
        tsne = TSNE(n_components=2, perplexity=perp, n_iter=800, random_state=42)
        embedded = tsne.fit_transform(filtered_features)

        plt.figure(figsize=(12, 8))
        unique_labels = sorted(set(filtered_labels))
        colors = plt.cm.tab10(np.linspace(0, 1, len(unique_labels)))

        for i, label in enumerate(unique_labels):
            lmask = np.array(filtered_labels) == label
            plt.scatter(embedded[lmask, 0], embedded[lmask, 1],
                       c=[colors[i]], label=label, s=30, alpha=0.6)

        plt.legend(fontsize=10, loc="best")
        plt.title("t-SNE of Visual Features (Color Histogram) by Verb Class")
        plt.xlabel("t-SNE Dim 1")
        plt.ylabel("t-SNE Dim 2")
        plt.tight_layout()
        plt.show()
    else:
        print("Not enough labeled frames for t-SNE visualization.")


In [ ]:
# t-SNE visualization of AUDIO features (MFCCs) segmented by action

def extract_segment_audio_features(audio_data, annotations_df, video_id, sr=22050):
    """
    Extract per-segment MFCC features by slicing audio according to annotation timestamps.
    Returns feature vectors and labels.
    """
    if video_id not in audio_data:
        return np.array([]), []

    y = audio_data[video_id]["audio"]
    vid_fps = 60.0  # EPIC-KITCHENS typical FPS

    vid_annot = annotations_df[annotations_df["video_id"] == video_id]

    features = []
    labels = []

    for _, row in vid_annot.iterrows():
        start_sec = row["start_frame"] / vid_fps
        stop_sec = row["stop_frame"] / vid_fps

        start_sample = int(start_sec * sr)
        stop_sample = int(stop_sec * sr)

        if stop_sample > len(y) or start_sample >= stop_sample:
            continue

        segment = y[start_sample:stop_sample]
        if len(segment) < 2048:
            continue

        # Compute MFCC for this segment
        mfcc = librosa.feature.mfcc(y=segment, sr=sr, n_mfcc=13)
        # Aggregate: mean and std across time
        mfcc_mean = mfcc.mean(axis=1)
        mfcc_std = mfcc.std(axis=1)
        feat_vec = np.concatenate([mfcc_mean, mfcc_std])

        features.append(feat_vec)
        labels.append(row["verb"])

    return np.array(features) if features else np.array([]), labels


# Extract per-segment audio features
all_audio_feats = []
all_audio_labels = []

for vid_id in video_ids:
    feats, labs = extract_segment_audio_features(audio_features, narrations_df, vid_id)
    if len(feats) > 0:
        all_audio_feats.append(feats)
        all_audio_labels.extend(labs)

if all_audio_feats:
    audio_feat_arr = np.vstack(all_audio_feats)
    print(f"Audio feature matrix shape: {audio_feat_arr.shape}")

    # t-SNE
    top_verbs = [v for v, _ in Counter(all_audio_labels).most_common(8)]
    mask = [l in top_verbs for l in all_audio_labels]
    filt_feats = audio_feat_arr[mask]
    filt_labels = [l for l, m in zip(all_audio_labels, mask) if m]

    if len(filt_feats) > 10:
        perp = min(30, len(filt_feats) - 1)
        tsne = TSNE(n_components=2, perplexity=perp, n_iter=800, random_state=42)
        emb = tsne.fit_transform(filt_feats)

        plt.figure(figsize=(12, 8))
        unique_labs = sorted(set(filt_labels))
        colors = plt.cm.tab10(np.linspace(0, 1, len(unique_labs)))
        for i, label in enumerate(unique_labs):
            lmask = np.array(filt_labels) == label
            plt.scatter(emb[lmask, 0], emb[lmask, 1],
                       c=[colors[i]], label=label, s=40, alpha=0.7)
        plt.legend(fontsize=10)
        plt.title("t-SNE of Audio Features (MFCC mean+std) by Verb Class")
        plt.xlabel("t-SNE Dim 1")
        plt.ylabel("t-SNE Dim 2")
        plt.tight_layout()
        plt.show()
else:
    print("No audio features available for t-SNE.")


In [ ]:
# Additional visualization: 2D histogram / hexbin of t-SNE embeddings
# This gives a density view rather than individual points

if len(visual_features_arr) > 10:
    perp = min(30, len(visual_features_arr) - 1)
    tsne_all = TSNE(n_components=2, perplexity=perp, n_iter=800, random_state=42)
    emb_all = tsne_all.fit_transform(visual_features_arr)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Hexbin density plot
    hb = axes[0].hexbin(emb_all[:, 0], emb_all[:, 1], gridsize=25, cmap='YlOrRd', mincnt=1)
    axes[0].set_title("t-SNE Density (Hexbin) — Visual Features")
    axes[0].set_xlabel("t-SNE Dim 1")
    axes[0].set_ylabel("t-SNE Dim 2")
    plt.colorbar(hb, ax=axes[0])

    # 2D KDE contour
    sns.kdeplot(x=emb_all[:, 0], y=emb_all[:, 1], ax=axes[1],
                fill=True, cmap="Blues", levels=15)
    axes[1].set_title("t-SNE Density (KDE Contour) — Visual Features")
    axes[1].set_xlabel("t-SNE Dim 1")
    axes[1].set_ylabel("t-SNE Dim 2")

    plt.tight_layout()
    plt.show()


---
## Part 4: Evaluation Metrics (20 pts)

### Writeup Questions

**1. What evaluation metrics are you planning on using? Why?**

We plan to use the following metrics, chosen to align with the EPIC-KITCHENS benchmark:
- **Top-1 Accuracy** (verb, noun, action): The primary metric — what fraction of predictions
  exactly match the ground truth. We compute this separately for verb, noun, and the
  combined action (verb+noun) since predicting both correctly is much harder.
- **Top-5 Accuracy**: Whether the correct label is in the model's top 5 predictions.
  Important because EPIC-KITCHENS has many fine-grained classes (97 verb, 300 noun),
  making exact prediction difficult.
- **Mean Class Accuracy**: Accuracy averaged per class rather than per sample. This
  accounts for class imbalance (e.g., "take" is far more common than "peel").

**2. Are there other metrics you considered?**

- **F1 Score (macro/weighted)**: Useful for imbalanced classification but less standard
  in the EPIC-KITCHENS literature.
- **Mean Average Precision (mAP)**: Standard for retrieval tasks in EPIC-KITCHENS but
  less applicable to our classification framing.
- **Confusion Matrix**: Not a scalar metric but invaluable for error analysis.

**3. Pros and cons:**

| Metric | Pros | Cons |
|--------|------|------|
| Top-1 Accuracy | Simple, interpretable, standard benchmark metric | Penalizes near-misses equally; biased toward majority class |
| Top-5 Accuracy | Captures "almost right" predictions; forgiving for fine-grained classes | Less precise; can hide poor discrimination |
| Mean Class Accuracy | Handles class imbalance; measures per-class fairness | Rare classes with few samples can be noisy; uncommon in papers |


In [ ]:
import numpy as np

def top_k_accuracy(predictions, ground_truths, k=1):
    """
    Compute top-k accuracy.

    Args:
        predictions (np.array): Predicted scores of shape (n_samples, n_classes).
                                Each row contains confidence scores for all classes.
        ground_truths (np.array): Ground truth class indices of shape (n_samples,).
        k (int): Number of top predictions to consider.

    Returns:
        float: Top-k accuracy.
    """
    if len(predictions) == 0:
        raise ValueError("No predictions provided!")
    if len(predictions) != len(ground_truths):
        raise ValueError(f"Mismatch: {len(predictions)} predictions vs {len(ground_truths)} ground truths")

    n_correct = 0
    for pred_scores, truth in zip(predictions, ground_truths):
        # Get indices of top-k predictions
        top_k_preds = np.argsort(pred_scores)[-k:]
        if truth in top_k_preds:
            n_correct += 1

    return n_correct / len(predictions)


def mean_class_accuracy(predictions, ground_truths, n_classes=None):
    """
    Compute mean per-class accuracy.

    Args:
        predictions (np.array): Predicted class indices of shape (n_samples,).
        ground_truths (np.array): Ground truth class indices of shape (n_samples,).
        n_classes (int): Total number of classes. Inferred if None.

    Returns:
        float: Mean class accuracy.
    """
    predictions = np.array(predictions)
    ground_truths = np.array(ground_truths)

    if n_classes is None:
        n_classes = max(ground_truths.max(), predictions.max()) + 1

    class_correct = np.zeros(n_classes)
    class_total = np.zeros(n_classes)

    for pred, truth in zip(predictions, ground_truths):
        class_total[truth] += 1
        if pred == truth:
            class_correct[truth] += 1

    # Only consider classes that appear in ground truth
    valid = class_total > 0
    per_class_acc = np.zeros(n_classes)
    per_class_acc[valid] = class_correct[valid] / class_total[valid]

    return per_class_acc[valid].mean()


def action_accuracy(verb_preds, noun_preds, verb_truths, noun_truths):
    """
    Compute action accuracy: both verb AND noun must be correct.

    Args:
        verb_preds (np.array): Predicted verb class indices.
        noun_preds (np.array): Predicted noun class indices.
        verb_truths (np.array): Ground truth verb class indices.
        noun_truths (np.array): Ground truth noun class indices.

    Returns:
        float: Action (verb+noun) accuracy.
    """
    n_correct = 0
    n_total = len(verb_preds)

    for vp, np_, vt, nt in zip(verb_preds, noun_preds, verb_truths, noun_truths):
        if vp == vt and np_ == nt:
            n_correct += 1

    return n_correct / n_total if n_total > 0 else 0.0


In [ ]:
# Test the metrics with synthetic data
np.random.seed(42)

n_samples = 200
n_verb_classes = 20
n_noun_classes = 50

# Simulate predictions (model output scores) and ground truths
verb_scores = np.random.randn(n_samples, n_verb_classes)
noun_scores = np.random.randn(n_samples, n_noun_classes)

# Ground truth: random class assignments
verb_gt = np.random.randint(0, n_verb_classes, n_samples)
noun_gt = np.random.randint(0, n_noun_classes, n_samples)

# Make predictions slightly better than random by boosting correct class
for i in range(n_samples):
    verb_scores[i, verb_gt[i]] += 2.0  # Boost correct verb
    noun_scores[i, noun_gt[i]] += 1.5  # Boost correct noun (slightly harder)

# Compute metrics
verb_top1 = top_k_accuracy(verb_scores, verb_gt, k=1)
verb_top5 = top_k_accuracy(verb_scores, verb_gt, k=5)
noun_top1 = top_k_accuracy(noun_scores, noun_gt, k=1)
noun_top5 = top_k_accuracy(noun_scores, noun_gt, k=5)

verb_preds = np.argmax(verb_scores, axis=1)
noun_preds = np.argmax(noun_scores, axis=1)

verb_mca = mean_class_accuracy(verb_preds, verb_gt, n_verb_classes)
noun_mca = mean_class_accuracy(noun_preds, noun_gt, n_noun_classes)
act_acc = action_accuracy(verb_preds, noun_preds, verb_gt, noun_gt)

print("=== Evaluation Metrics (Synthetic Test) ===")
print(f"Verb  Top-1: {verb_top1:.3f}  |  Top-5: {verb_top5:.3f}  |  Mean Class Acc: {verb_mca:.3f}")
print(f"Noun  Top-1: {noun_top1:.3f}  |  Top-5: {noun_top5:.3f}  |  Mean Class Acc: {noun_mca:.3f}")
print(f"Action (verb+noun) Accuracy: {act_acc:.3f}")
print()
print("Note: Action accuracy is much lower since both verb AND noun must be correct.")


---
## Part 5: Instruction Tuning Prompts (15 pts)

Instruction tuning constrains model output through careful prompt design.
Below are prompts for each scenario that guide a model to produce specific,
structured outputs.


### Scenario 1: Restaurant Review Sentiment

**Review:** "This place stinks, the service was awful and the food was not cooked. I will never come back here!"

**Prompt:**
```
You are a sentiment classifier. Given a restaurant review, respond with exactly
one word: "positive", "negative", or "neutral". Do not include any explanation
or additional text.

Review: "This place stinks, the service was awful and the food was not cooked.
I will never come back here!"

Sentiment:
```

**Expected output:** `negative`


### Scenario 2: Facial Emotion Classification

**Prompt:**
```
You are an emotion classifier for facial expressions. You will be shown an image
of a person's face. Classify the emotion being expressed using exactly one of the
following labels: "angry", "sad", or "happy". Respond with only the label and
nothing else.

[IMAGE]

Emotion:
```

**Expected output:** One of `angry`, `sad`, or `happy`


### Scenario 3: Entity/Information Extraction from Novels

**Paragraph:** "The man, Edgar, flew to Italy to hike the Alps. He was looking forward to going skiing there."

**Prompt for subject name:**
```
Extract the name of the main subject from the following paragraph.
Respond with only the name, nothing else.

Paragraph: "The man, Edgar, flew to Italy to hike the Alps. He was looking forward
to going skiing there."

Name:
```
**Expected output:** `Edgar`

**Prompt for destination:**
```
Where is the subject traveling to in the following paragraph?
Respond with only the destination country or location, nothing else.

Paragraph: "The man, Edgar, flew to Italy to hike the Alps. He was looking forward
to going skiing there."

Destination:
```
**Expected output:** `Italy`

**Prompt for planned activities:**
```
List the activities the subject plans to do in the following paragraph.
Respond with only a comma-separated list of activities, nothing else.

Paragraph: "The man, Edgar, flew to Italy to hike the Alps. He was looking forward
to going skiing there."

Activities:
```
**Expected output:** `hiking, skiing`


---
## Summary

In this notebook, we:
1. Downloaded a subset of EPIC-KITCHENS-100 (annotations + videos from P01)
2. Extracted four modalities: RGB frames, audio (spectrograms/MFCCs), optical flow, and text narrations
3. Visualized data distributions, aligned samples, and t-SNE embeddings of features
4. Implemented evaluation metrics: Top-k accuracy, mean class accuracy, and action accuracy
5. Designed instruction tuning prompts for three scenarios

This dataset and preprocessing pipeline will be carried forward into subsequent homework
assignments for model training and evaluation.
